# Embedding Space Visualization

Visualizing how argument sentences cluster in embedding space across different datasets and labels.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from umap import UMAP

sys.path.insert(0, str(Path.cwd().parent.parent))
from gaic.embeddings import get_collection

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "figure.dpi": 150,
})

In [ ]:
collection = get_collection()
print(f"Collection: {collection.count()} documents")

data = collection.get(include=["embeddings", "metadatas"])
embeddings = np.array(data["embeddings"])
ids = data["ids"]
labels = [m["label"] for m in data["metadatas"]]
datasets = [m["dataset"] for m in data["metadatas"]]

df = pd.DataFrame({
    "id": ids,
    "label": labels,
    "dataset": datasets,
})
print(f"Embedding shape: {embeddings.shape}")
df["dataset"].value_counts()

In [ ]:
print("Fitting UMAP (this may take a minute)...")
reducer = UMAP(n_components=2, n_neighbors=30, min_dist=0.3, metric="cosine", random_state=42)
embedding_2d = reducer.fit_transform(embeddings)
df["x"] = embedding_2d[:, 0]
df["y"] = embedding_2d[:, 1]
print("Done.")

## Plot 1: All Datasets — Color by Dataset

Shows how sentences from different datasets cluster in embedding space. Distinct clusters suggest domain-specific language patterns.

In [ ]:
DATASET_ORDER = ["ABSTRCT", "ACQUA", "AEC", "AFS", "ARGUMINSCI", "FINARG", "IAM", "PE", "SCIARK", "USELEC"]
COLORS = plt.cm.tab10.colors
dataset_colors = {ds: COLORS[i] for i, ds in enumerate(DATASET_ORDER)}

fig, ax = plt.subplots(figsize=(10, 8))

for ds in DATASET_ORDER:
    mask = df["dataset"] == ds
    ax.scatter(
        df.loc[mask, "x"], df.loc[mask, "y"],
        c=[dataset_colors[ds]], label=ds, alpha=0.6, s=12, edgecolors="none"
    )

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title("Embedding Space: 10 Argument Mining Datasets")
ax.legend(loc="upper right", framealpha=0.9, markerscale=2)
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.savefig("fig_embeddings_by_dataset.pdf", bbox_inches="tight")
plt.savefig("fig_embeddings_by_dataset.png", bbox_inches="tight", dpi=300)
plt.show()

## Plot 2: ABSTRCT Dataset — Color by Label

Zooms into the highest-performing dataset (ABSTRCT) to visualize separation between Argument and No-Argument classes.

In [ ]:
df_abstrct = df[df["dataset"] == "ABSTRCT"].copy()
print(f"ABSTRCT samples: {len(df_abstrct)}")

label_colors = {"Argument": "#2ecc71", "No-Argument": "#e74c3c"}

fig, ax = plt.subplots(figsize=(8, 7))

for label in ["No-Argument", "Argument"]:
    mask = df_abstrct["label"] == label
    ax.scatter(
        df_abstrct.loc[mask, "x"], df_abstrct.loc[mask, "y"],
        c=label_colors[label], label=label, alpha=0.7, s=25, edgecolors="white", linewidths=0.3
    )

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title("ABSTRCT: Argument vs No-Argument")
ax.legend(loc="upper right", framealpha=0.9, markerscale=1.5)
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.savefig("fig_abstrct_by_label.pdf", bbox_inches="tight")
plt.savefig("fig_abstrct_by_label.png", bbox_inches="tight", dpi=300)
plt.show()

## Plot 3: Dataset-Label Interaction

Small multiples showing Argument/No-Argument separation per dataset. Reveals which datasets have clear embedding-space separation (easier for k-NN) vs mixed distributions.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6), sharex=True, sharey=True)
axes = axes.flatten()

for i, ds in enumerate(DATASET_ORDER):
    ax = axes[i]
    df_ds = df[df["dataset"] == ds]
    
    for label in ["No-Argument", "Argument"]:
        mask = df_ds["label"] == label
        ax.scatter(
            df_ds.loc[mask, "x"], df_ds.loc[mask, "y"],
            c=label_colors[label], alpha=0.5, s=8, edgecolors="none"
        )
    
    ax.set_title(ds, fontsize=10, fontweight="bold")
    ax.set_xticks([])
    ax.set_yticks([])

handles = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=label_colors["Argument"], markersize=8, label="Argument"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=label_colors["No-Argument"], markersize=8, label="No-Argument"),
]
fig.legend(handles=handles, loc="lower center", ncol=2, framealpha=0.9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Argument vs No-Argument Separation by Dataset", y=1.02, fontsize=13)

plt.tight_layout()
plt.savefig("fig_facet_by_dataset_label.pdf", bbox_inches="tight")
plt.savefig("fig_facet_by_dataset_label.png", bbox_inches="tight", dpi=300)
plt.show()

## Plot 4: Dataset Centroids & Distances

Shows the centroid of each dataset and visualizes inter-dataset distances. Helps explain cross-dataset transfer potential.

In [ ]:
centroids = df.groupby("dataset")[["x", "y"]].mean()

fig, ax = plt.subplots(figsize=(10, 8))

for ds in DATASET_ORDER:
    mask = df["dataset"] == ds
    ax.scatter(
        df.loc[mask, "x"], df.loc[mask, "y"],
        c=[dataset_colors[ds]], alpha=0.15, s=8, edgecolors="none"
    )

for ds in DATASET_ORDER:
    cx, cy = centroids.loc[ds, "x"], centroids.loc[ds, "y"]
    ax.scatter(cx, cy, c=[dataset_colors[ds]], s=200, edgecolors="black", linewidths=1.5, marker="o", zorder=10)
    ax.annotate(ds, (cx, cy), fontsize=9, fontweight="bold", ha="center", va="bottom",
                xytext=(0, 8), textcoords="offset points")

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title("Dataset Centroids in Embedding Space")
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.savefig("fig_dataset_centroids.pdf", bbox_inches="tight")
plt.savefig("fig_dataset_centroids.png", bbox_inches="tight", dpi=300)
plt.show()

## Plot 5: Domain Centroids

Groups datasets by their domain cluster (Scientific, Political/Debate, Professional, Educational) and shows how domains occupy distinct regions in embedding space.

In [ ]:
# Domain mapping based on thesis description
DOMAIN_MAP = {
    "ABSTRCT": "Scientific",
    "ARGUMINSCI": "Scientific", 
    "SCIARK": "Scientific",
    "AEC": "Political/Debate",
    "AFS": "Political/Debate",
    "USELEC": "Political/Debate",
    "FINARG": "Professional",
    "ACQUA": "Professional",
    "PE": "Educational",
    "IAM": "Educational",
}

DOMAIN_ORDER = ["Scientific", "Political/Debate", "Professional", "Educational"]
DOMAIN_COLORS = {
    "Scientific": "#3498db",       # Blue
    "Political/Debate": "#e74c3c", # Red
    "Professional": "#2ecc71",     # Green
    "Educational": "#9b59b6",      # Purple
}

df["domain"] = df["dataset"].map(DOMAIN_MAP)

fig, ax = plt.subplots(figsize=(10, 8))

# Plot all points faded by domain
for domain in DOMAIN_ORDER:
    mask = df["domain"] == domain
    ax.scatter(
        df.loc[mask, "x"], df.loc[mask, "y"],
        c=DOMAIN_COLORS[domain], alpha=0.15, s=8, edgecolors="none"
    )

# Plot dataset centroids colored by domain
for ds in DATASET_ORDER:
    domain = DOMAIN_MAP[ds]
    cx, cy = centroids.loc[ds, "x"], centroids.loc[ds, "y"]
    ax.scatter(cx, cy, c=DOMAIN_COLORS[domain], s=200, edgecolors="black", linewidths=1.5, marker="o", zorder=10)
    ax.annotate(ds, (cx, cy), fontsize=9, fontweight="bold", ha="center", va="bottom",
                xytext=(0, 8), textcoords="offset points")

# Legend for domains
handles = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=DOMAIN_COLORS[d], 
               markersize=10, markeredgecolor="black", markeredgewidth=1, label=d)
    for d in DOMAIN_ORDER
]
ax.legend(handles=handles, loc="upper right", framealpha=0.9, title="Domain")

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title("Dataset Centroids by Domain Cluster")
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.savefig("fig_domain_centroids.pdf", bbox_inches="tight")
plt.savefig("fig_domain_centroids.png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# Compute domain centroids and inter-domain distances
domain_centroids = df.groupby("domain")[["x", "y"]].mean().loc[DOMAIN_ORDER]

domain_centroid_matrix = domain_centroids.values
domain_dist_matrix = squareform(pdist(domain_centroid_matrix, metric="euclidean"))
df_domain_dist = pd.DataFrame(domain_dist_matrix, index=DOMAIN_ORDER, columns=DOMAIN_ORDER)

print("=== Inter-Domain Centroid Distances (UMAP space) ===")
print(df_domain_dist.round(2).to_string())

# Intra-domain spread (how tight is each domain cluster?)
print("\n=== Intra-Domain Spread (std of dataset centroids within domain) ===")
for domain in DOMAIN_ORDER:
    domain_datasets = [ds for ds, d in DOMAIN_MAP.items() if d == domain]
    domain_dataset_centroids = centroids.loc[domain_datasets].values
    spread = np.std(pdist(domain_dataset_centroids)) if len(domain_datasets) > 1 else 0
    print(f"{domain:20s}: {spread:.2f} ({', '.join(domain_datasets)})")

## Summary Statistics

In [ ]:
from scipy.spatial.distance import pdist, squareform

centroid_matrix = centroids.loc[DATASET_ORDER].values
dist_matrix = squareform(pdist(centroid_matrix, metric="euclidean"))
df_dist = pd.DataFrame(dist_matrix, index=DATASET_ORDER, columns=DATASET_ORDER)

print("=== Inter-Dataset Centroid Distances (UMAP space) ===")
print(df_dist.round(2).to_string())

In [ ]:
from scipy.spatial.distance import cdist

separation_scores = []
for ds in DATASET_ORDER:
    df_ds = df[df["dataset"] == ds]
    arg_pts = df_ds[df_ds["label"] == "Argument"][["x", "y"]].values
    noarg_pts = df_ds[df_ds["label"] == "No-Argument"][["x", "y"]].values
    
    arg_centroid = arg_pts.mean(axis=0)
    noarg_centroid = noarg_pts.mean(axis=0)
    
    inter_class_dist = np.linalg.norm(arg_centroid - noarg_centroid)
    arg_spread = np.std(cdist(arg_pts, [arg_centroid]))
    noarg_spread = np.std(cdist(noarg_pts, [noarg_centroid]))
    avg_spread = (arg_spread + noarg_spread) / 2
    
    separation_scores.append({
        "dataset": ds,
        "inter_class_dist": inter_class_dist,
        "avg_class_spread": avg_spread,
        "separation_ratio": inter_class_dist / avg_spread if avg_spread > 0 else 0,
    })

df_sep = pd.DataFrame(separation_scores).sort_values("separation_ratio", ascending=False)
print("=== Class Separation Scores (higher = more separable) ===")
print(df_sep.to_string(index=False))